# BPDR panel demonstration: input, output and evidence

**Goal:** preserve a usable vocal idea, offer only supported improvements, and explain when another recording is needed. An unusual performance is not automatically a mistake. Ordinary speech does not automatically contain a recoverable intended melody.

This short walkthrough uses completed, checksum-verified experiments. It shows a controlled pitch change, a separate noise-reduction preview, and a correct no-change outcome. It also displays failed real-voice learning gates. **Acoustic engineering checks are not musical approval or proof of the singer's intention.**

Run with **BPDR local Python** using `Run-Notebooks.ps1`, then **Restart Kernel and Run All Cells**. Execution reads evidence, recomputes small summaries and runs one CPU forward pass; it performs no training, dataset extraction, review adoption or prepared-audio handoff.

For the full record see [training](03_transformer_training.ipynb), [evaluation](04_transformer_evaluation.ipynb), and [data/experiment history](02_data_review_and_paired_experiments.ipynb).

In [ ]:
import os, sys, importlib.util
from pathlib import Path
start = Path(os.environ.get('BPDR_COMPONENT_ROOT', Path.cwd())).resolve()
COMPONENT = next((p for p in (start, *start.parents)
    if p.name == 'bpdr_input' and (p/'pyproject.toml').is_file()), None)
if COMPONENT is None:
    raise RuntimeError('Open inside BPDR_Workspace using Run-Notebooks.ps1.')
WORKSPACE = COMPONENT/'BPDR_Workspace'
sys.path.insert(0, str(WORKSPACE/'source'))
required = ['numpy', 'pandas', 'matplotlib', 'soundfile', 'torch', 'IPython']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing '+', '.join(missing)+'. Select '+str(COMPONENT/'.venv/Scripts/python.exe'))
import json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import display, Audio, Markdown, Image
from bpdr_input.reporting.notebook_evidence import load_index, component_path
from bpdr_input.application.saved_results import pitch_example, noise_example

def read(path): return json.loads(Path(path).read_text(encoding='utf-8'))
def sha(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

print('Interpreter:', sys.executable)
print('Workspace:', WORKSPACE)
print('Execution scope: no optimizer updates, no dataset extraction.')

## 1. Verify what is being demonstrated

The working model was pretrained on constructed phrases, then adapted using Vocadito/Jamendo pairs plus replay. Injected local shifts give known engineering targets; they do not label an unrestricted singer's intended correct melody. The checkpoint remains experimental and prepared handoff remains disabled.

The assertions below stop the walkthrough if selected evidence or checkpoint bytes have changed.

In [ ]:
INDEX, EVIDENCE = load_index(COMPONENT)
M = read(EVIDENCE['presentation_manifest'])
assert sha(EVIDENCE['presentation_manifest']) == INDEX['current_gate']['presentation_manifest_sha256']
for path, digest in M['source_hashes'].items():
    assert sha(path) == digest, path
CARD = read(EVIDENCE['presentation_manifest'].parent/'model_card.json')
assert sha(EVIDENCE['presentation_manifest'].parent/'model_card.json') == M['model_card_sha256']
working_card = read(WORKSPACE/'selected_model/model.json')
CHECKPOINT = WORKSPACE/'selected_model'/working_card['checkpoint']
CHECKPOINT_BEFORE = sha(CHECKPOINT)
assert CHECKPOINT_BEFORE == working_card['sha256'] == CARD['checkpoint_sha256']
assert not working_card['research_model_adopted'] and not working_card['prepared_handoff_enabled']
ADAPT = read(EVIDENCE['presentation_adapt'])
display(pd.DataFrame([{
    'checkpoint': str(CHECKPOINT), 'sha256': CHECKPOINT_BEFORE,
    'training': working_card['training'], 'research_adopted': working_card['research_model_adopted'],
    'prepared_handoff_enabled': working_card['prepared_handoff_enabled']}]))
print('Verified', len(M['source_hashes']), 'saved evidence files and the selected checkpoint.')

## 2. Research gap and four-view mechanism

A correction system can appear stable by predicting zero everywhere. BPDR therefore asks for two behaviors together: **remain stable when boundary metadata changes, and respond when the audio is deliberately detuned**. Repeated boundary variants are dependent versions of a phrase, not additional singers.

| View | Waveform | Boundary metadata | Known engineering target |
| --- | --- | --- | --- |
| A | Original | Nominal | Zero alteration |
| B | Same original | Split, merge or jitter | Zero alteration |
| C | Known local detuning added | Nominal | Opposite of the added detuning |
| D | Same detuned waveform | Same changed metadata as B | Same as C |

The next plot displays an actual saved training example. The CPU forward pass then loads the selected checkpoint and returns framewise correction proposals. A finite output confirms executable inference, not accuracy. Targets are used for training/scoring, never provided as reference audio to the runtime model.

In [ ]:
from bpdr_input.experiments.learning.controlled_study import load_pair
from bpdr_input.experiments.learning.fine_pitch_relative_check import relative_pair
pretrain_dir = Path(M['synthetic_runs']['339'])
pretrain_decl = read(pretrain_dir/'declaration.json')
catalog = read(Path(pretrain_decl['data'])/'catalog.json')
record = next(r for r in catalog['records'] if r['split']=='train' and r['plateau_frames']>=10)
pair = relative_pair(load_pair(record))
fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
for view in 'AC': axes[0].plot(pair['inputs'][view][:,64], label=view, alpha=.8)
for view in 'AB': axes[1].plot(pair['inputs'][view][:,70], label=view, alpha=.8)
axes[2].plot(-pair['detuning'], label='C/D compensating target', color='#9d4e35')
axes[0].set(ylabel='Relative pitch (cents)', title='Actual saved four-view training example: '+pair['pair_id'])
axes[1].set(ylabel='Boundary score')
axes[2].set(xlabel='Aligned frame index', ylabel='Correction (cents)')
for ax in axes: ax.legend(loc='upper right')
plt.tight_layout(); plt.show()
assert np.array_equal(pair['inputs']['A'][:,:70], pair['inputs']['B'][:,:70])
assert np.array_equal(pair['inputs']['C'][:,:70], pair['inputs']['D'][:,:70])
print('Verified: boundary changes leave acoustic features identical within A/B and C/D.')

In [ ]:
import torch
from bpdr_input.reporting.bpdr_showcase import load_model
from bpdr_input.modeling.examples import collate
torch.set_num_threads(2)
model, normalizer, width = load_model(CHECKPOINT)
model.eval()
batch = collate([pair], normalizer, width)
with torch.inference_mode():
    prediction = {view: model(x, batch['padding']) for view, x in batch['inputs'].items()}
assert all(torch.isfinite(x).all().item() for x in prediction.values())
assert model.parameter_count() == CARD['architecture']['parameters']
display(pd.DataFrame([model.describe()]))
print('Input shape:', tuple(batch['inputs']['A'].shape), 'Output shape:', tuple(prediction['A'].shape))
print('All outputs finite; checkpoint unchanged:', sha(CHECKPOINT)==CHECKPOINT_BEFORE)

## 3. Controlled pitch demonstration: hear the original, input and output

The reference is a constructed phrase. A known local pitch shift was injected to make the input; the selected model generated the processed preview. The original reference is available for scoring only and was not an inference input. All players use the same gain (`normalize=False`).

Compare the three players, then inspect the measured local pitch error. A successful selected example demonstrates a local engineering behavior; it is not population accuracy, novice-singer validation or proof of BPDR superiority. The processed file is a preview; the accepted downstream waveform remains the original input.

In [ ]:
pitch = pitch_example()
assert pitch['accepted_downstream_unchanged']
pitch_waves = [sf.read(pitch[key], dtype='float64') for key in ['reference','input','output']]
assert len({rate for _,rate in pitch_waves}) == 1
assert len({wave.shape for wave,_ in pitch_waves}) == 1
for label, key in [('Reference: before deliberate detuning','reference'),
                   ('Input: deliberately detuned phrase','input'),
                   ('Output: model-generated processed preview','output')]:
    display(Markdown('**'+label+'**'))
    display(Audio(filename=str(pitch[key]), normalize=False))
error_in, error_out = abs(pitch['before_cents']), abs(pitch['after_cents'])
display(pd.DataFrame([{'input_local_error_cents': error_in, 'output_local_error_cents': error_out,
    'single_example_error_reduction': 1-error_out/error_in,
    'scope': pitch['scope']}]))
fig, ax = plt.subplots(figsize=(7,3.5))
ax.bar(['Input','Processed preview'], [error_in,error_out], color=['#a75b40','#387b68'])
ax.set(ylabel='Absolute local error (cents)', title='One known-reference example: 100 cents = one semitone')
plt.tight_layout(); plt.show()
print('Verified audio grid:', pitch_waves[0][1], 'Hz;', len(pitch_waves[0][0]), 'samples.')

## 4. Separate real-source demonstration: humming plus added rain

This starts with a real HumTrans humming recording and mixes it with known recorded rain. Conservative deterministic spectral attenuation uses a **caller-declared noise-only prefix**. This is a separate audio-preparation method, not Transformer denoising.

The baseline before noise mixing permits a paired SNR calculation. The displayed SNR gain measures this constructed noise mixture, not naturally co-recorded noisy novice recordings. Numerical pitch preservation does not certify breaths, consonants, listening quality or intended notes.

In [ ]:
noise = noise_example()
assert noise['accepted_downstream_unchanged']
for label,key in [('Input: humming with added recorded rain','input'),
                  ('Output: conservative noise-reduction preview','output')]:
    display(Markdown('**'+label+'**'))
    display(Audio(filename=str(noise[key]), normalize=False))
display(pd.DataFrame([{'paired_snr_gain_db': noise['snr_gain_db'],
    'noise_prefix_attenuation_db': noise['reference_attenuation_db'],
    'median_pitch_change_cents': noise['median_pitch_change_cents'],
    'scope': noise['scope']}]))
x,sr = sf.read(noise['input'], dtype='float64')
y,rate = sf.read(noise['output'], dtype='float64')
assert sr==rate and x.shape==y.shape
ix=np.arange(0,min(len(x),int(.7*sr)),10)
fig,ax=plt.subplots(figsize=(11,3.5))
ax.plot(ix/sr,x[ix],label='Input',alpha=.65)
ax.plot(ix/sr,y[ix],label='Processed preview',alpha=.8)
ax.set(xlabel='Seconds',ylabel='Amplitude (same scale)',title='Declared noise-only prefix: no timing change')
ax.legend();plt.tight_layout();plt.show()
print('Sample rate, duration and sample count preserved; accepted handoff remains unchanged.')

## 5. No-change is a useful outcome

An acceptable original phrase does not need an audible difference. The saved runtime returned `IDENTITY` for this undetuned constructed original and `ACCEPT_UNCHANGED` for its technical handoff. We compare decoded samples below, since WAV headers can differ while the waveform stays identical.

This verifies preservation of this example. It does not establish that the system recognizes every correct melody or knows the singer's intended scale.

In [ ]:
runtime_rows = read(COMPONENT/'runs/presentation_runtime_checks_mixed.json')
identity_row = next(r for r in runtime_rows if r['label']=='mixed_synthetic_original_audio_only')
identity_dir = Path(identity_row['path'])
identity = read(identity_dir/'model_report.json')
assert identity['outcome']=='IDENTITY' and not identity['prepared_handoff_enabled']
accepted = identity['handoff']['accepted_audio']
assert sha(accepted['path']) == accepted['checksum']
assert sha(identity['source']) == sha(identity_dir/'original.wav')
a, ar = sf.read(identity['source'],dtype='float64')
b, br = sf.read(accepted['path'],dtype='float64')
assert ar==br and np.array_equal(a,b)
for label, path in [('Original input',identity['source']),('Accepted output: same decoded waveform',accepted['path'])]:
    display(Markdown('**'+label+'**'));display(Audio(filename=str(path),normalize=False))
display(pd.DataFrame([{'model_outcome':identity['outcome'], 'handoff_status':identity['handoff']['status'],
    'sample_count':len(a), 'maximum_sample_difference':float(np.max(np.abs(a-b))),
    'decoded_samples_identical':np.array_equal(a,b)}]))

## 6. Failed real-voice learning must remain visible

On singer/artist-group-separated real development examples, the selected model's paired response stayed near zero. An always-zero model can preserve originals but cannot recover known detuning. The later bounded real/humming attempts also failed declared checks and were not adopted. Larger automatic fitting was stopped after its prerequisite failed.

**Why this matters:** a lower training loss or a favorable constructed example cannot substitute for real-singer repair. The table uses saved dependent perturbation variants, not counts of independent successful participants. Development material has been examined repeatedly; it is not an untouched final test.

The recorded engineering gate requires response ≥0.25, original alteration ≤1 cent, and positive correction gain. Those thresholds are historical study checks, not validated musical-usability thresholds.

In [ ]:
real_rows=[]
for dataset, metrics in ADAPT['results']['bpdr']['by_dataset'].items():
    real_rows.append(dict(stage='Selected model: adapted real development',dataset=dataset,**metrics))
humming=read(EVIDENCE['humming_learning'])
for split in ['humming_training','humming_development']:
    real_rows.append(dict(stage='Later candidate: '+split,dataset='humtrans',**humming['results'][split]['all']))
real=pd.DataFrame(real_rows)
real['engineering_gate_pass']=(real.median_response_gain>=.25)&(real.unshifted_alteration<=1)&(real.correction_gain>0)
display(real[['stage','dataset','records','correction_mae','zero_mae','median_response_gain',
    'unshifted_alteration','correction_gain','engineering_gate_pass']])
fig,axes=plt.subplots(1,2,figsize=(12,4))
labels=['Vocadito dev','Jamendo dev','HumTrans train','HumTrans dev']
# Match labels to explicit datasets rather than dictionary insertion order.
order=[next(i for i,r in enumerate(real_rows) if r['dataset']==d and 'Selected model' in r['stage']) for d in ['vocadito','jamendo']]+[2,3]
axes[0].bar(labels,real.iloc[order].median_response_gain,color=['#467c89','#467c89','#bd8950','#a65349'])
axes[0].axhline(.25,color='grey',ls=':',label='Historical response gate')
axes[0].set(ylabel='Paired response gain (ideal 1)');axes[0].legend(fontsize=8)
axes[1].bar(labels,real.iloc[order].unshifted_alteration,color=['#467c89','#467c89','#bd8950','#a65349'])
axes[1].axhline(1,color='grey',ls=':',label='Historical preservation limit')
axes[1].set(ylabel='Original alteration (cents)');axes[1].legend(fontsize=8)
for ax in axes: ax.tick_params(axis='x',rotation=15)
plt.tight_layout();plt.show()
decision=read(COMPONENT/'runs/completion_model_decision.json')
print('Recorded decision:',decision['decision'])
print('Larger fit:',decision['larger_v2_matched_fit'])
print('Humming gates:',decision['humming_gates'])
assert not decision['new_v2_real_training_gate_passes']
assert not decision['research_model_adopted'] and not decision['prepared_handoff_enabled']

## 7. What this component can currently decide

| Question | Demonstrated behavior | Remaining limit |
| --- | --- | --- |
| Can the file be processed? | Format, duration, silence/clipping and valid-sample checks | Technical acceptance does not prove usable singing |
| Can it preserve an original? | Unchanged waveform case above; originals retained during previews | Broader preservation needs independent listening |
| Can it show a supported local pitch change? | Known-reference constructed example above | Generalizing real-singer repair remains unestablished |
| Can it reduce eligible added noise? | Declared-reference preview above with remeasurement | Natural noisy novice recordings need evaluation |
| Was a strange note intentional? | No validated intention classifier | Key/scale alone cannot identify the intended note |
| Can ordinary reading become the intended melody? | No validated capability | Request a sung/hummed take when musical evidence is insufficient |
| Does prepared audio improve the next component? | Versioned handoff and fixed-extractor diagnostics exist | Independent downstream benefit is not yet demonstrated |

The practical target is **accept unchanged when useful, accept a verified preparation when validated, or request a more usable take with a concrete reason**. Melodic recoverability and intention-aware musical approval remain research work. Do not force a scale, flatten expressive slides, or loosen a gate to create a visible output difference.

For a live panel input, use a short solo hum/sung phrase. Explain that it may be preserved, produce a separate preview, or have its proposed edit withheld. A valid model result, a controller withholding an edit, and a rendering failure are different outcomes.

## 8. Verify the assembled panel evidence package

The manifest links the reusable demo audio, plots and source receipts. Every packaged audio item is checked by checksum, sample grid and duration before display. Source receipts are also checked, so labels cannot silently drift away from the saved evidence. The package may include additional technical failure/withholding cases; these demonstrate handling, not an intention classifier.

In [ ]:
manifest_path=WORKSPACE/'results/panel_demo/manifest.json'
if not manifest_path.is_file():
    print('Panel package is not generated yet. Run the component panel-demo builder, then rerun this cell.')
    PANEL_MANIFEST_VERIFIED=False
else:
    from bpdr_input.reporting.panel_demo import load_verified_manifest
    panel=load_verified_manifest(manifest_path,verify_sources=True)
    display(pd.DataFrame([{'case':case['id'],'title':case['title'],'source_kind':case['source_kind'],
        'outcome':case['outcome'],'handoff_status':case.get('handoff_status'),
        'has_output':bool(case.get('output'))} for case in panel['cases']]))
    print('Manifest:',manifest_path)
    print('Manifest SHA-256:',sha(manifest_path))
    print('Package scope:',panel.get('scope'))
    for limitation in panel.get('limitations',[]): print('-',limitation)
    PANEL_MANIFEST_VERIFIED=True

## 9. Panel explanation and reproducibility

Suggested explanation: “The research question is whether a pitch-correction model can ignore segmentation mistakes while still responding to genuine acoustic detuning. The four aligned views test both. This prototype preserves originals, shows separately measured previews, and records failed cases. The constructed pitch example and real-source noise example are working engineering demonstrations. We have not established general correction of arbitrary novice singing, musical intention inference, or automatic prepared-audio acceptance.”

The presentation should keep the input/output comparison beside its scope, and keep the failed real-data gate visible. If a live run cannot finish, show its error receipt and use these verified saved examples without calling them a fresh successful run.

Before broader use: declare a melodic-usability protocol; collect reviewed singing, speech-like singing, ordinary speech and failed recordings; test on held-out singers; measure unnecessary edits and rerecording requests; evaluate the fixed downstream extractor and user/listener preservation judgments. Keep the original/no-processing and conservative-rule baselines.

**Completion check below:** no fitting, no checkpoint changes, no automatic promotion of a preview.

In [ ]:
assert sha(CHECKPOINT)==CHECKPOINT_BEFORE
assert not working_card['prepared_handoff_enabled']
print('Selected checkpoint unchanged:',CHECKPOINT_BEFORE)
print('Panel package verified:',PANEL_MANIFEST_VERIFIED)
print('No training, no dataset extraction and no prepared handoff were performed.')
print('Audio players: 3 pitch + 2 noise + 2 unchanged; plots: four-view, pitch error, noise prefix, failed real gates.')